# RNA 3D Structure Prediction - Kaggle Inference Notebook (v2.0)

This notebook is optimized for creating Kaggle submissions from our trained model. It handles:

1. Loading the trained model
2. Processing test data in the Kaggle format
3. Running inference with the dual-mode approach (no dihedral angles at test time)
4. Formatting the predictions for Kaggle submission

## Kaggle Environment Setup

This notebook automatically detects whether it's running in the Kaggle environment and adjusts paths accordingly.
For Kaggle submissions, you need to set up the following datasets:

1. **stanford-rna-3d-folding**: Contains the test_sequences.csv file
2. **rna-3d-features**: Contains the preprocessed feature files
3. **rna-3d-models**: Contains the trained model files
4. **rna-model-src**: Contains the source code for the model implementation

The notebook will save the final predictions as `submission.csv` in the Kaggle working directory.

## Memory Optimizations

This notebook includes multiple memory optimizations for the Kaggle P100 GPU:

1. Memory allocator configuration to reduce fragmentation
2. Adaptive batch size based on sequence length
3. Mixed precision inference with FP16
4. Explicit memory cleanup between operations
5. Enhanced positional encoding that dynamically extends for long sequences

## Key Features

- Handles corrupted model checkpoints
- Processes multi-structure feature files
- Extended positional encoding for long sequences
- Memory-efficient processing for Kaggle's GPU constraints
- Automatic path detection for Kaggle environment

In [1]:
# Memory optimization for Kaggle P100 GPU
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,max_split_size_mb:512"

# Basic imports
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import math  # Needed for positional encoding
from pathlib import Path
from tqdm.notebook import tqdm

# Detect Kaggle environment
is_kaggle = os.path.exists('/kaggle')

# Add project root to path for module imports
import sys
if is_kaggle:
    # Add Kaggle-specific module path
    module_path = "/kaggle/input/rna-model-src"
    if module_path not in sys.path:
        sys.path.append(module_path)
    print(f"Added Kaggle module path: {module_path}")
else:
    # Local environment module path
    module_path = os.path.abspath(os.path.join('..'))
    if module_path not in sys.path:
        sys.path.append(module_path)
    print(f"Added local module path: {module_path}")

# Import project modules
from src.models.rna_folding_model import RNAFoldingModel
from src.data_loading import RNADataset, collate_fn, create_data_loader

# Memory monitoring function
def log_memory_usage(step_name=""):
    """Log current GPU memory usage."""
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / 1024**2
        reserved = torch.cuda.memory_reserved() / 1024**2
        free, total = torch.cuda.mem_get_info()
        free = free / 1024**2
        total = total / 1024**2
        print(f"Memory at {step_name}: Allocated: {allocated:.1f}MB, Reserved: {reserved:.1f}MB, Free: {free:.1f}MB, Total: {total:.1f}MB")

# Apply data loading patch for multi-structure feature files
def fixed_load_precomputed_features(
    target_id, features_dir, temporal_cutoff=None
):
    """
    Enhanced version of load_precomputed_features that handles inconsistent feature file formats.
    """
    import os
    import numpy as np
    import warnings
    import pandas as pd
    
    features = {}

    # 1. Load dihedral features
    dihedral_path = os.path.join(
        features_dir, "dihedral_features", f"{target_id}_dihedral_features.npz"
    )
    if os.path.exists(dihedral_path):
        try:
            with np.load(dihedral_path) as data:
                # Check feature generation date if available for temporal cutoff
                if temporal_cutoff is not None and "metadata" in data:
                    try:
                        metadata_str = str(data["metadata"])
                        if "extraction_timestamp" in metadata_str:
                            timestamp_part = metadata_str.split("extraction_timestamp")[
                                1
                            ].split("'")[1]
                            generation_date = timestamp_part.split()[0]

                            if pd.to_datetime(generation_date) > pd.to_datetime(
                                temporal_cutoff
                            ):
                                warnings.warn(
                                    f"Dihedral features for {target_id} were generated after the temporal cutoff. Using zeros."
                                )
                                features["dihedral"] = None
                                return features
                    except (KeyError, IndexError, ValueError):
                        pass

                # Handle different feature file formats
                if "features" in data:
                    # Standard format
                    features["dihedral"] = {"features": data["features"].astype(np.float32)}
                elif "struct_1_features" in data:
                    # Multi-structure format with numbered structures
                    features["dihedral"] = {"features": data["struct_1_features"].astype(np.float32)}
                else:
                    # Unknown format - warn and use None
                    warnings.warn(f"Dihedral features file for {target_id} has unexpected format. Using zeros.")
                    features["dihedral"] = None
                    return features
                
                # Handle NaN values if present
                if features["dihedral"] is not None and np.isnan(features["dihedral"]["features"]).any():
                    features["dihedral"]["features"] = np.nan_to_num(
                        features["dihedral"]["features"], nan=0.0
                    )
        except Exception as e:
            # Handle any errors in loading
            warnings.warn(f"Error loading dihedral features for {target_id}: {str(e)}. Using zeros.")
            features["dihedral"] = None
    else:
        # For test data or if file is missing
        features["dihedral"] = None
        warnings.warn(f"Dihedral features not found for {target_id}. Using zeros.")

    # 2. Load thermodynamic features (required)
    thermo_path = os.path.join(
        features_dir, "thermo_features", f"{target_id}_thermo_features.npz"
    )
    if not os.path.exists(thermo_path):
        raise ValueError(
            f"Thermodynamic features not found for {target_id}. Required for prediction."
        )

    try:
        with np.load(thermo_path) as data:
            # Check feature generation date if available for temporal cutoff
            if temporal_cutoff is not None and "generation_date" in data:
                generation_date = str(data["generation_date"])
                if pd.to_datetime(generation_date) > pd.to_datetime(temporal_cutoff):
                    warnings.warn(
                        f"Thermo features for {target_id} were generated after the temporal cutoff. Using zeros."
                    )
                    features["thermo"] = None
                    return features

            # Extract key arrays and scalar values
            thermo_features = {}

            # Get pairing probabilities matrix (critical)
            if "pairing_probs" in data:
                thermo_features["pairing_probs"] = data["pairing_probs"].astype(np.float32)
            elif "base_pair_probs" in data:
                thermo_features["pairing_probs"] = data["base_pair_probs"].astype(np.float32)
            else:
                raise ValueError(f"No pairing probabilities found in {target_id} thermo features")

            # Handle NaN values in pairing probabilities
            if np.isnan(thermo_features["pairing_probs"]).any():
                thermo_features["pairing_probs"] = np.nan_to_num(
                    thermo_features["pairing_probs"], nan=0.0
                )

            # Get positional entropy (optional)
            if "positional_entropy" in data:
                thermo_features["positional_entropy"] = data["positional_entropy"].astype(
                    np.float32
                )
            else:
                # Calculate from pairing probabilities if missing
                pair_probs = thermo_features["pairing_probs"]
                row_entropies = -np.sum(
                    pair_probs * np.log2(pair_probs + 1e-10), axis=1
                )
                thermo_features["positional_entropy"] = row_entropies

            # Get accessibility (optional)
            if "accessibility" in data:
                thermo_features["accessibility"] = data["accessibility"].astype(np.float32)
            else:
                # Calculate from pairing probabilities if missing
                pair_probs = thermo_features["pairing_probs"]
                accessibilities = 1.0 - np.sum(pair_probs, axis=1)
                thermo_features["accessibility"] = np.maximum(0.0, accessibilities)

            features["thermo"] = thermo_features
    except Exception as e:
        raise ValueError(f"Error loading thermodynamic features for {target_id}: {str(e)}")

    # 3. Load evolutionary coupling features (optional)
    mi_path = os.path.join(
        features_dir, "evolutionary_features", f"{target_id}_evolutionary_features.npz"
    )
    if os.path.exists(mi_path):
        try:
            with np.load(mi_path) as data:
                # Check feature generation date if available for temporal cutoff
                if temporal_cutoff is not None and "generation_date" in data:
                    generation_date = str(data["generation_date"])
                    if pd.to_datetime(generation_date) > pd.to_datetime(temporal_cutoff):
                        warnings.warn(
                            f"Evolutionary features for {target_id} were generated after the temporal cutoff. Using zeros."
                        )
                        features["evolutionary"] = None
                        return features

                # Extract key arrays and metadata
                evol_features = {}

                # Get coupling matrix (required for this feature type)
                if "coupling_matrix" in data:
                    coupling_matrix = data["coupling_matrix"].astype(np.float32)
                    
                    # Check if the matrix is valid (not all zeros or constant)
                    is_valid = not np.allclose(coupling_matrix, 0.0)
                    evol_features["has_valid_mi"] = is_valid
                    
                    if is_valid:
                        evol_features["coupling_matrix"] = coupling_matrix
                    else:
                        # Zero matrix case - still provide the matrix but flag it
                        evol_features["coupling_matrix"] = coupling_matrix
                        warnings.warn(f"Coupling matrix for {target_id} is all zeros or constant.")
                else:
                    # No coupling matrix found
                    evol_features["has_valid_mi"] = False
                    evol_features["coupling_matrix"] = None

                features["evolutionary"] = evol_features
        except Exception as e:
            # Handle errors in evolutionary feature loading
            warnings.warn(f"Error loading evolutionary features for {target_id}: {str(e)}. Proceeding without them.")
            features["evolutionary"] = None
    else:
        # No evolutionary features available
        features["evolutionary"] = None

    return features

# Apply the patch
from src import data_loading
original_load_precomputed_features = data_loading.load_precomputed_features
data_loading.load_precomputed_features = fixed_load_precomputed_features
print("Successfully patched data_loading.load_precomputed_features with fixed version")

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
log_memory_usage("Initial")

Added local module path: /home/smcgee/MLprojects/RNA_2025/Pipeline-v1-March27on/RNA-2025-Pipeline-v0.2.0-BetaBend
Successfully patched data_loading.load_precomputed_features with fixed version
Using device: cuda
Memory at Initial: Allocated: 0.0MB, Reserved: 0.0MB, Free: 12653.6MB, Total: 15931.9MB


## 1. Configuration

Set paths and parameters for inference

In [2]:
# Detect Kaggle environment
is_kaggle = os.path.exists('/kaggle')
print(f"Running in Kaggle environment: {is_kaggle}")

# Paths based on environment
if is_kaggle:
    # Kaggle environment paths
    TEST_SEQUENCES_PATH = "/kaggle/input/stanford-rna-3d-folding/test_sequences.csv"
    FEATURES_DIR = "/kaggle/input/rna-3d-features/"
    OUTPUT_DIR = "/kaggle/working/"
    
    # Model paths in Kaggle
    MODEL_PATHS = {
        "final_model": "/kaggle/input/rna-3d-models/best_model.pt",
        "production_run_1": "/kaggle/input/rna-3d-models/production_model.pt",
    }
else:
    # Local environment paths
    TEST_SEQUENCES_PATH = "../data/raw/test_sequences.csv"
    FEATURES_DIR = "../data/processed/"
    OUTPUT_DIR = "../submissions/"
    
    # Local model paths
    MODEL_PATHS = {
        "final_model": "../results/final_model/run_20250423-072601/checkpoints/best_model.pt",
        "tuning_lr_0.001": "../results/tuning_run_1/lr_0.001/run_20250423-072437/checkpoints/best_model.pt",
        "tuning_lr_0.0005": "../results/tuning_run_1/lr_0.0005/run_20250423-072448/checkpoints/best_model.pt",
        "tuning_lr_0.0001": "../results/tuning_run_1/lr_0.0001/run_20250423-072458/checkpoints/best_model.pt",
        "production_run_1": "../results/production_run_1/run_20250423-072209/checkpoints/best_model.pt"
    }

# Choose which model to use (set to None to evaluate all and pick the best)
SELECTED_MODEL = "final_model" if is_kaggle else None  # For Kaggle, always use the final model
USE_ENSEMBLE = False  # Set to True to use model ensembling

# Create output directory
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Parameters
BATCH_SIZE = 8
NUM_SAMPLES = 5  # For Kaggle, we need 5 conformations per sequence
TEMPERATURE = 0.1  # Temperature for sampling diversity

# Set the May 2022 temporal cutoff date - critical for proper evaluation
TEMPORAL_CUTOFF = "2022-05-01"

print(f"Test sequences path: {TEST_SEQUENCES_PATH}")
print(f"Features directory: {FEATURES_DIR}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Selected model: {SELECTED_MODEL}")

Running in Kaggle environment: False
Test sequences path: ../data/raw/test_sequences.csv
Features directory: ../data/processed/
Output directory: ../submissions/
Selected model: None


## 2. Load Model

Load the trained model from checkpoint. 

**Note:** This code includes a fix for corrupted checkpoints that contain only a 'dummy' key in model_state_dict.
The fix initializes models from scratch using the correct architecture from the checkpoint, allowing
inference to proceed even when weights can't be loaded.

In [3]:
def load_model(checkpoint_path):
    """Load model from checkpoint with robustness to corrupted state_dict."""
    print(f"Loading model from {checkpoint_path}")
    
    # Check if the file exists
    if not os.path.exists(checkpoint_path):
        print(f"ERROR: Checkpoint file not found at {checkpoint_path}")
        return None, None, {'val_rmsd': None, 'epoch': None}
    
    try:
        # Load checkpoint
        checkpoint = torch.load(checkpoint_path, map_location=device)
        
        # Get model configuration
        if 'args' in checkpoint:
            config = checkpoint['args']
        elif 'model_config' in checkpoint:
            config = checkpoint['model_config']
        else:
            # Default configuration
            print("WARNING: No configuration found in checkpoint. Using defaults.")
            config = {
                'num_blocks': 4,
                'residue_embed_dim': 128,
                'pair_embed_dim': 64,
                'num_attention_heads': 4,
                'dropout': 0.1
            }
        
        # Initialize model
        model = RNAFoldingModel(config)
        
        # Check if state_dict is valid
        if 'model_state_dict' in checkpoint:
            model_state_dict = checkpoint['model_state_dict']
            # Check if it's corrupted (only contains dummy key)
            if list(model_state_dict.keys()) == ['dummy']:
                print("WARNING: Corrupted checkpoint detected with only 'dummy' key.")
                print("Initializing model from scratch with the configuration from checkpoint.")
                # We don't load weights - using freshly initialized weights
            else:
                # Load state dict if it seems valid
                model.load_state_dict(model_state_dict)
                print("Successfully loaded weights from checkpoint.")
        elif 'state_dict' in checkpoint:
            model.load_state_dict(checkpoint['state_dict'])
            print("Successfully loaded weights from 'state_dict'.")
        else:
            print("WARNING: No state_dict found in checkpoint. Using untrained model.")
        
        # Move to device
        model = model.to(device)
        model.eval()
        
        # Extract validation metrics if available
        val_rmsd = None
        if 'val_rmsd' in checkpoint:
            val_rmsd = checkpoint['val_rmsd']
        elif 'validation_rmsd' in checkpoint:
            val_rmsd = checkpoint['validation_rmsd']
        elif 'best_val_metrics' in checkpoint and 'rmsd' in checkpoint['best_val_metrics']:
            val_rmsd = checkpoint['best_val_metrics']['rmsd']
        
        epoch = None
        if 'epoch' in checkpoint:
            epoch = checkpoint['epoch']
        
        # Enhance model to handle longer sequences
        patch_model_for_long_sequences(model)
        
        # Return model, config, and metrics
        return model, config, {'val_rmsd': val_rmsd, 'epoch': epoch}
    
    except Exception as e:
        print(f"ERROR loading model: {str(e)}")
        import traceback
        traceback.print_exc()
        return None, None, {'val_rmsd': None, 'epoch': None}

# Import necessary PyTorch modules
import torch.nn as nn  # Add missing import for nn

# Enhanced positional encoding that can handle longer sequences
class EnhancedPositionalEncoding(nn.Module):
    """
    Enhanced version of PositionalEncoding that handles sequences longer than max_len.
    """
    
    def __init__(self, config):
        """Initialize positional encoding with extendable length."""
        super().__init__()

        # Extract parameters from config
        self.embed_dim = config.get("residue_embed_dim", 128)
        self.max_len = config.get("max_len", 500)

        # Create constant positional encoding matrix
        position = torch.arange(0, self.max_len).unsqueeze(1).float()
        self.div_term = torch.exp(
            torch.arange(0, self.embed_dim, 2).float() * (-math.log(10000.0) / self.embed_dim)
        )

        pe = torch.zeros(self.max_len, self.embed_dim)
        pe[:, 0::2] = torch.sin(position * self.div_term)
        pe[:, 1::2] = torch.cos(position * self.div_term)

        # Register buffer (not a parameter, but part of state)
        self.register_buffer("pe", pe.unsqueeze(0))  # Shape: (1, max_len, embed_dim)
    
    def extend_pe(self, new_max_len):
        """Dynamically extend the positional encoding to handle longer sequences."""
        # Create new positions
        old_max_len = self.pe.size(1)
        if new_max_len <= old_max_len:
            return  # No need to extend
            
        print(f"Extending positional encoding from {old_max_len} to {new_max_len}")
        
        # Generate positions for the new entries
        position = torch.arange(old_max_len, new_max_len).unsqueeze(1).float().to(self.pe.device)
        
        # Create new encodings
        pe_extension = torch.zeros(new_max_len - old_max_len, self.embed_dim, 
                                  device=self.pe.device)
        pe_extension[:, 0::2] = torch.sin(position * self.div_term.to(self.pe.device))
        pe_extension[:, 1::2] = torch.cos(position * self.div_term.to(self.pe.device))
        
        # Concatenate with existing buffer
        new_pe = torch.cat([self.pe.squeeze(0), pe_extension], dim=0).unsqueeze(0)
        
        # Replace the buffer
        self.pe = new_pe
        self.max_len = new_max_len
    
    def forward(self, seq_len):
        """Get positional encodings with automatic extension if needed."""
        if seq_len > self.max_len:
            # If sequence is longer than our current max, extend the encoding
            new_max_len = max(seq_len, int(self.max_len * 1.5))  # Grow by 50% to reduce frequent extensions
            self.extend_pe(new_max_len)
            
        return self.pe[:, :seq_len]

def patch_model_for_long_sequences(model):
    """Patch a model's positional encoding to handle long sequences."""
    if hasattr(model, 'embedding_module') and hasattr(model.embedding_module, 'positional_encoding'):
        # Get original module
        orig_pe = model.embedding_module.positional_encoding
        
        # Create config for new module
        config = {
            'residue_embed_dim': orig_pe.embed_dim,
            'max_len': orig_pe.max_len
        }
        
        # Create enhanced module
        enhanced_pe = EnhancedPositionalEncoding(config)
        
        # Copy the existing buffer
        enhanced_pe.pe = orig_pe.pe.clone()
        
        # Replace in the model
        model.embedding_module.positional_encoding = enhanced_pe
        
        print(f"Patched model with enhanced positional encoding (max_len: {enhanced_pe.max_len})")
        return True
    else:
        print("Warning: Could not find positional encoding in model structure")
        return False

def evaluate_models(model_paths, valid_loader=None):
    """Evaluate multiple models to select the best one.
    
    Args:
        model_paths: Dictionary of model paths
        valid_loader: Optional validation data loader
        
    Returns:
        Dictionary of loaded models with their metrics
    """
    models = {}
    
    # Load all models
    for name, path in model_paths.items():
        try:
            if os.path.exists(path):
                model, config, metrics = load_model(path)
                
                # Only add if model loaded successfully
                if model is not None:
                    models[name] = {
                        'model': model,
                        'config': config,
                        'metrics': metrics,
                        'path': path
                    }
                    
                    print(f"Loaded {name}:")
                    print(f"  Validation RMSD: {metrics['val_rmsd']}")
                    print(f"  Trained epochs: {metrics['epoch']}")
                    print()
            else:
                print(f"WARNING: Model path not found: {path}")
        except Exception as e:
            print(f"ERROR loading model {name}: {str(e)}")
    
    # If no validation loader, select the model with the best validation metrics
    if valid_loader is None:
        sorted_models = sorted(
            [(name, info) for name, info in models.items() if info['metrics']['val_rmsd'] is not None],
            key=lambda x: x[1]['metrics']['val_rmsd']
        )
        
        if sorted_models:
            best_name, best_info = sorted_models[0]
            print(f"\nBest model based on validation RMSD: {best_name}")
            print(f"Validation RMSD: {best_info['metrics']['val_rmsd']}")
            print(f"Trained epochs: {best_info['metrics']['epoch']}")
        else:
            print("\nWARNING: No models with validation metrics found")
    
    return models

# Load models
try:
    if SELECTED_MODEL is not None and SELECTED_MODEL in MODEL_PATHS:
        # Load only the selected model
        model_path = MODEL_PATHS[SELECTED_MODEL]
        model, config, metrics = load_model(model_path)
        
        if model is not None:
            models = {
                SELECTED_MODEL: {
                    'model': model,
                    'config': config,
                    'metrics': metrics,
                    'path': model_path
                }
            }
            print(f"Using selected model: {SELECTED_MODEL}")
        else:
            models = {}
            print(f"ERROR: Failed to load selected model: {SELECTED_MODEL}")
    else:
        # Evaluate all models
        models = evaluate_models(MODEL_PATHS)
        
        # Choose the best model if not using ensemble
        if not USE_ENSEMBLE:
            # Sort by validation RMSD
            sorted_models = sorted(
                [(name, info) for name, info in models.items() if info['metrics']['val_rmsd'] is not None],
                key=lambda x: x[1]['metrics']['val_rmsd']
            )
            
            if sorted_models:
                best_name, best_info = sorted_models[0]
                # Keep only the best model
                models = {best_name: best_info}
                print(f"Selected best model: {best_name}")
            else:
                print("WARNING: No models with validation metrics. Using the first available model.")
                if models:
                    first_name = list(models.keys())[0]
                    models = {first_name: models[first_name]}
                    print(f"Using model: {first_name}")
                else:
                    # Handle the case when no models loaded successfully
                    print("ERROR: No valid models were loaded.")
                    models = {}
    
    print(f"Using {len(models)} model(s) for inference")
    
except Exception as e:
    print(f"ERROR during model loading: {str(e)}")
    import traceback
    traceback.print_exc()
    models = {}

Loading model from ../results/final_model/run_20250423-072601/checkpoints/best_model.pt
Initializing model from scratch with the configuration from checkpoint.
Patched model with enhanced positional encoding (max_len: 500)
Loaded final_model:
  Validation RMSD: 7.593239451519988
  Trained epochs: 30

Loading model from ../results/tuning_run_1/lr_0.001/run_20250423-072437/checkpoints/best_model.pt
Initializing model from scratch with the configuration from checkpoint.
Patched model with enhanced positional encoding (max_len: 500)
Loaded tuning_lr_0.001:
  Validation RMSD: 12.711866538487572
  Trained epochs: 10

Loading model from ../results/tuning_run_1/lr_0.0005/run_20250423-072448/checkpoints/best_model.pt
Initializing model from scratch with the configuration from checkpoint.
Patched model with enhanced positional encoding (max_len: 500)
Loaded tuning_lr_0.0005:
  Validation RMSD: 12.711866538487572
  Trained epochs: 10

Loading model from ../results/tuning_run_1/lr_0.0001/run_20250

/tmp/ipykernel_107521/4293364273.py:12: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(checkpoint_path, map_location=device)


## 3. Load Test Data

Create data loader for test sequences

In [4]:
def create_adaptive_test_loader(sequences_path, features_dir, batch_size=8, temporal_cutoff=None):
    """Create test data loader with adaptive batch size based on sequence length."""
    # Check if files exist
    if not os.path.exists(sequences_path):
        print(f"ERROR: Test sequences file not found at {sequences_path}")
        return None, None
    
    if not os.path.exists(features_dir):
        print(f"ERROR: Features directory not found at {features_dir}")
        return None, None
    
    try:
        # Create dataset
        dataset = RNADataset(
            sequences_csv_path=sequences_path,
            features_dir=features_dir,
            temporal_cutoff=temporal_cutoff,
            use_validation_set=True,  # For test time
            require_features=False,  # Allow sequences without all features
        )
        
        # Get sequence lengths to determine optimal batch size
        seq_lengths = []
        for i in range(min(10, len(dataset))):  # Sample a few sequences to check length
            sample = dataset[i]
            if 'sequence' in sample:
                seq_lengths.append(len(sample['sequence']))
            elif 'sequence_int' in sample:
                seq_lengths.append(len(sample['sequence_int']))
        
        # Read more sequences if we haven't found any long ones
        if max(seq_lengths) < 500 and len(dataset) > 10:
            for i in range(10, min(30, len(dataset))):
                sample = dataset[i]
                if 'sequence' in sample:
                    seq_lengths.append(len(sample['sequence']))
                elif 'sequence_int' in sample:
                    seq_lengths.append(len(sample['sequence_int']))
        
        # Determine adaptive batch size based on longest sequence
        max_length = max(seq_lengths)
        
        # P100-optimized batch size selection
        if max_length < 250:
            adaptive_batch_size = 8
        elif max_length < 400:
            adaptive_batch_size = 4
        elif max_length < 600:
            adaptive_batch_size = 2
        else:
            adaptive_batch_size = 1
        
        print(f"Using adaptive batch size {adaptive_batch_size} for max sequence length {max_length}")
        log_memory_usage("Before DataLoader creation")
        
        # Create data loader with the adaptive batch size
        loader = torch.utils.data.DataLoader(
            dataset,
            batch_size=adaptive_batch_size,
            shuffle=False,
            collate_fn=collate_fn,
            num_workers=4,
        )
        
        print(f"Test loader created with {len(dataset)} sequences, batch size {adaptive_batch_size}")
        return loader, adaptive_batch_size
    except Exception as e:
        print(f"ERROR creating test loader: {str(e)}")
        import traceback
        traceback.print_exc()
        return None, None

# Create test loader with adaptive batch size
try:
    log_memory_usage("Before test loader creation")
    test_loader, BATCH_SIZE = create_adaptive_test_loader(
        TEST_SEQUENCES_PATH, 
        FEATURES_DIR, 
        BATCH_SIZE,  # Initial batch size, will be adapted
        TEMPORAL_CUTOFF
    )
    
    if test_loader is None:
        print("WARNING: Failed to create test loader. Inference cannot continue.")
    else:
        print(f"Adapted batch size to {BATCH_SIZE} based on sequence lengths")
    
    log_memory_usage("After test loader creation")
except Exception as e:
    print(f"ERROR: {str(e)}")
    import traceback
    traceback.print_exc()
    test_loader = None

Memory at Before test loader creation: Allocated: 20.8MB, Reserved: 22.0MB, Free: 12631.6MB, Total: 15931.9MB
Using adaptive batch size 1 for max sequence length 720
Memory at Before DataLoader creation: Allocated: 20.8MB, Reserved: 22.0MB, Free: 12631.6MB, Total: 15931.9MB
Test loader created with 12 sequences, batch size 1
Adapted batch size to 1 based on sequence lengths
Memory at After test loader creation: Allocated: 20.8MB, Reserved: 22.0MB, Free: 12631.6MB, Total: 15931.9MB


## 4. Run Inference

Run inference on test data with multiple samples per sequence

In [5]:
def generate_samples(model, batch, num_samples=5, temperature=0.1):
    """Generate multiple structure samples for each sequence in the batch.
    
    Args:
        model: RNA folding model
        batch: Input batch
        num_samples: Number of structure samples to generate
        temperature: Temperature for sampling diversity
        
    Returns:
        List of dictionaries containing samples
    """
    results = []
    
    # Log memory usage before moving batch to device
    log_memory_usage(f"Before processing batch of size {len(batch['target_ids'])}")
    
    # Get batch device and size
    batch_device = {k: v.to(device) if isinstance(v, torch.Tensor) else v 
                   for k, v in batch.items()}
    
    # Log tensor shapes to understand memory usage
    for k, v in batch_device.items():
        if isinstance(v, torch.Tensor):
            print(f"INFO: {k} shape: {v.shape}")
    
    # Generate samples
    for i in range(num_samples):
        # Forward pass with noise scale controlled by temperature
        with torch.no_grad(), torch.cuda.amp.autocast():  # Use mixed precision for memory efficiency
            # Set dropout for diverse sampling
            model.train()  # Enable dropout for diverse sampling
            
            # Log memory before forward pass
            log_memory_usage(f"Before forward pass (sample {i+1}/{num_samples})")
            
            try:
                outputs = model(batch_device)
                
                # Log memory after forward pass
                log_memory_usage(f"After forward pass (sample {i+1}/{num_samples})")
                
                model.eval()  # Restore eval mode
                
                # Collect results
                for j, target_id in enumerate(batch['target_ids']):
                    seq_len = batch['lengths'][j].item()
                    
                    # Get coordinates and confidence
                    coords = outputs['pred_coords'][j, :seq_len].cpu().numpy()
                    conf = torch.sigmoid(outputs['pred_confidence'][j, :seq_len]).cpu().numpy()
                    
                    # Add to results
                    results.append({
                        'target_id': target_id,
                        'sample_id': i,
                        'coords': coords,
                        'confidence': conf,
                    })
                
                # Clear memory between samples
                torch.cuda.empty_cache()
                
            except RuntimeError as e:
                # If we run out of memory, try to recover
                if "CUDA out of memory" in str(e):
                    print(f"WARNING: Out of memory generating sample {i+1}. Trying to recover...")
                    torch.cuda.empty_cache()
                    
                    # Retry with even smaller batch or more aggressive memory optimization
                    # This would need custom logic if needed
                    raise e
                else:
                    raise e
    
    return results

def run_inference(model, data_loader, num_samples=5, temperature=0.1):
    """Run inference on all test data.
    
    Args:
        model: RNA folding model
        data_loader: Test data loader
        num_samples: Number of structure samples to generate per sequence
        temperature: Temperature for sampling diversity
        
    Returns:
        Dictionary mapping target_id to list of sample dictionaries
    """
    all_results = {}
    
    # Process all batches
    for batch_idx, batch in enumerate(tqdm(data_loader, desc="Running inference")):
        # Log memory before processing batch
        log_memory_usage(f"Before batch {batch_idx+1}/{len(data_loader)}")
        
        # Generate samples
        try:
            samples = generate_samples(model, batch, num_samples, temperature)
            
            # Organize by target_id
            for sample in samples:
                target_id = sample['target_id']
                if target_id not in all_results:
                    all_results[target_id] = []
                all_results[target_id].append(sample)
            
            # Log memory after batch completion
            log_memory_usage(f"After batch {batch_idx+1}/{len(data_loader)}")
            
            # Clear memory between batches
            torch.cuda.empty_cache()
            
        except RuntimeError as e:
            # If we run out of memory, try to recover and skip the problematic batch
            if "CUDA out of memory" in str(e):
                print(f"ERROR: Out of memory on batch {batch_idx+1}. Skipping...")
                torch.cuda.empty_cache()
                continue
            else:
                raise e
    
    return all_results

In [6]:
# Define ensemble inference function
def run_ensemble_inference(models_dict, data_loader, num_samples=5, temperature=0.1):
    """Run inference using an ensemble of models.
    
    Args:
        models_dict: Dictionary of models
        data_loader: Test data loader
        num_samples: Number of structure samples to generate per sequence
        temperature: Temperature for sampling diversity
        
    Returns:
        Dictionary mapping target_id to list of sample dictionaries
    """
    all_results = {}
    model_weights = {}
    
    # Check if we have any models to work with
    if not models_dict:
        print("ERROR: No models available for inference")
        return all_results
    
    # Determine weights based on validation RMSD (lower RMSD = higher weight)
    valid_models = [(name, info) for name, info in models_dict.items() 
                   if info['metrics']['val_rmsd'] is not None]
    
    if valid_models:
        # Invert RMSD to get weights (lower RMSD = higher weight)
        rmsd_values = [info['metrics']['val_rmsd'] for _, info in valid_models]
        min_rmsd = min(rmsd_values)
        
        # Calculate weights (normalized)
        for name, info in valid_models:
            rmsd = info['metrics']['val_rmsd']
            # Use inverse squared RMSD for more pronounced weighting
            model_weights[name] = (min_rmsd / rmsd) ** 2
        
        # Normalize weights
        weight_sum = sum(model_weights.values())
        if weight_sum > 0:
            for name in model_weights:
                model_weights[name] /= weight_sum
    else:
        # Equal weights if no validation metrics
        for name in models_dict:
            model_weights[name] = 1.0 / len(models_dict)
    
    print("Model ensemble weights:")
    for name, weight in model_weights.items():
        print(f"  {name}: {weight:.4f}")
    
    # For each batch in the dataloader
    for batch_idx, batch in enumerate(tqdm(data_loader, desc="Running ensemble inference")):
        # Log memory usage before batch
        log_memory_usage(f"Before ensemble batch {batch_idx+1}/{len(data_loader)}")
        
        # Get batch device
        batch_device = {k: v.to(device) if isinstance(v, torch.Tensor) else v 
                       for k, v in batch.items()}
        
        # Generate samples from each model
        all_model_samples = {}
        
        # First pass: collect predictions from all models
        for name, model_info in models_dict.items():
            model = model_info['model']
            weight = model_weights.get(name, 0.0)
            
            # Skip model if weight is too low (optional optimization)
            if weight < 0.05:
                continue
                
            try:
                # Generate samples for this model with mixed precision
                with torch.cuda.amp.autocast():  # Use mixed precision for memory efficiency
                    model_samples = generate_samples(model, batch_device, num_samples, temperature)
                
                # Organize by target_id and sample_id
                for sample in model_samples:
                    target_id = sample['target_id']
                    sample_id = sample['sample_id']
                    
                    if target_id not in all_model_samples:
                        all_model_samples[target_id] = {}
                    
                    if sample_id not in all_model_samples[target_id]:
                        all_model_samples[target_id][sample_id] = []
                    
                    # Store sample with its weight
                    all_model_samples[target_id][sample_id].append({
                        'coords': sample['coords'],
                        'confidence': sample['confidence'],
                        'weight': weight
                    })
            except RuntimeError as e:
                # Handle OOM errors gracefully
                if "CUDA out of memory" in str(e):
                    print(f"WARNING: OOM with model {name}. Skipping this model for the current batch.")
                    torch.cuda.empty_cache()
                    continue
                else:
                    raise e
        
        # Second pass: combine predictions with weighted averaging
        for target_id, samples_dict in all_model_samples.items():
            if target_id not in all_results:
                all_results[target_id] = []
            
            for sample_id, model_preds in samples_dict.items():
                # Get first prediction to determine shape
                first_pred = model_preds[0]
                combined_coords = np.zeros_like(first_pred['coords'])
                combined_conf = np.zeros_like(first_pred['confidence'])
                total_weight = 0.0
                
                # Weighted average of coordinates and confidence
                for pred in model_preds:
                    weight = pred['weight']
                    combined_coords += pred['coords'] * weight
                    combined_conf += pred['confidence'] * weight
                    total_weight += weight
                
                # Normalize by total weight
                if total_weight > 0:
                    combined_coords /= total_weight
                    combined_conf /= total_weight
                
                # Add combined prediction to results
                all_results[target_id].append({
                    'target_id': target_id,
                    'sample_id': sample_id,
                    'coords': combined_coords,
                    'confidence': combined_conf
                })
        
        # Log memory after batch and clear cache
        log_memory_usage(f"After ensemble batch {batch_idx+1}/{len(data_loader)}")
        torch.cuda.empty_cache()
    
    return all_results

# Run inference - either with a single model or ensemble
log_memory_usage("Before starting inference")

if USE_ENSEMBLE and len(models) > 1:
    print("Running ensemble inference with multiple models...")
    results = run_ensemble_inference(models, test_loader, NUM_SAMPLES, TEMPERATURE)
else:
    # Check if we have any models
    if not models:
        print("ERROR: No models available for inference. Check model paths and loading.")
        results = {}  # Return empty results to avoid errors in subsequent cells
    else:
        # Get the first (and only) model from the dictionary
        model_name = list(models.keys())[0]
        model = models[model_name]['model']
        print(f"Running inference with single model: {model_name}")
        results = run_inference(model, test_loader, NUM_SAMPLES, TEMPERATURE)

log_memory_usage("After inference completion")

Memory at Before starting inference: Allocated: 20.8MB, Reserved: 22.0MB, Free: 12629.5MB, Total: 15931.9MB
Running inference with single model: final_model


Running inference:   0%|          | 0/12 [00:00<?, ?it/s]

Memory at Before batch 1/12: Allocated: 20.8MB, Reserved: 22.0MB, Free: 12629.5MB, Total: 15931.9MB
Memory at Before processing batch of size 1: Allocated: 20.8MB, Reserved: 22.0MB, Free: 12629.5MB, Total: 15931.9MB
INFO: lengths shape: torch.Size([1])
INFO: sequence_int shape: torch.Size([1, 69])
INFO: dihedral_features shape: torch.Size([1, 69, 4])
INFO: dihedral_targets shape: torch.Size([1, 69, 4])
INFO: pairing_probs shape: torch.Size([1, 69, 69])
INFO: positional_entropy shape: torch.Size([1, 69])
INFO: accessibility shape: torch.Size([1, 69])
INFO: coupling_matrix shape: torch.Size([1, 69, 69])
INFO: conservation shape: torch.Size([1, 69])
INFO: mask shape: torch.Size([1, 69])
Memory at Before forward pass (sample 1/5): Allocated: 20.9MB, Reserved: 22.0MB, Free: 12629.5MB, Total: 15931.9MB


/tmp/ipykernel_107521/1435751240.py:30: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.no_grad(), torch.cuda.amp.autocast():  # Use mixed precision for memory efficiency
INFO: pairing_probs shape: torch.Size([1, 69, 69])
INFO: coupling_matrix shape: torch.Size([1, 69, 69])
INFO: rel_pos_batch shape: torch.Size([1, 69, 69, 32])
INFO: pair_features_list[0] shape: torch.Size([1, 69, 69, 1])
INFO: pair_features_list[1] shape: torch.Size([1, 69, 69, 1])
INFO: pair_features_list[2] shape: torch.Size([1, 69, 69, 32])
INFO: pairing_probs shape: torch.Size([1, 69, 69])
INFO: coupling_matrix shape: torch.Size([1, 69, 69])
INFO: rel_pos_batch shape: torch.Size([1, 69, 69, 32])
INFO: pair_features_list[0] shape: torch.Size([1, 69, 69, 1])
INFO: pair_features_list[1] shape: torch.Size([1, 69, 69, 1])
INFO: pair_features_list[2] shape: torch.Size([1, 69, 69, 32])
INFO: pairing_probs shape: torch.Size([1, 69, 69]

Memory at After forward pass (sample 1/5): Allocated: 30.9MB, Reserved: 66.0MB, Free: 12537.5MB, Total: 15931.9MB
Memory at Before forward pass (sample 2/5): Allocated: 29.0MB, Reserved: 44.0MB, Free: 12559.5MB, Total: 15931.9MB
Memory at After forward pass (sample 2/5): Allocated: 30.9MB, Reserved: 66.0MB, Free: 12537.5MB, Total: 15931.9MB
Memory at Before forward pass (sample 3/5): Allocated: 29.0MB, Reserved: 44.0MB, Free: 12559.5MB, Total: 15931.9MB
Memory at After forward pass (sample 3/5): Allocated: 30.9MB, Reserved: 66.0MB, Free: 12537.5MB, Total: 15931.9MB
Memory at Before forward pass (sample 4/5): Allocated: 29.0MB, Reserved: 44.0MB, Free: 12559.5MB, Total: 15931.9MB
Memory at After forward pass (sample 4/5): Allocated: 30.9MB, Reserved: 66.0MB, Free: 12537.5MB, Total: 15931.9MB
Memory at Before forward pass (sample 5/5): Allocated: 29.0MB, Reserved: 44.0MB, Free: 12559.5MB, Total: 15931.9MB
Memory at After forward pass (sample 5/5): Allocated: 30.9MB, Reserved: 66.0MB, Free

...

Memory at Before forward pass (sample 2/5): Allocated: 33.1MB, Reserved: 44.0MB, Free: 12565.5MB, Total: 15931.9MB
Memory at After forward pass (sample 2/5): Allocated: 35.0MB, Reserved: 4908.0MB, Free: 7701.5MB, Total: 15931.9MB
Memory at Before forward pass (sample 3/5): Allocated: 33.1MB, Reserved: 44.0MB, Free: 12565.5MB, Total: 15931.9MB
Memory at After forward pass (sample 3/5): Allocated: 35.0MB, Reserved: 4908.0MB, Free: 7701.5MB, Total: 15931.9MB


INFO: pairing_probs shape: torch.Size([1, 720, 720])
INFO: coupling_matrix shape: torch.Size([1, 720, 720])
INFO: rel_pos_batch shape: torch.Size([1, 720, 720, 32])
INFO: pair_features_list[0] shape: torch.Size([1, 720, 720, 1])
INFO: pair_features_list[1] shape: torch.Size([1, 720, 720, 1])
INFO: pair_features_list[2] shape: torch.Size([1, 720, 720, 32])
INFO: pairing_probs shape: torch.Size([1, 720, 720])
INFO: coupling_matrix shape: torch.Size([1, 720, 720])
INFO: rel_pos_batch shape: torch.Size([1, 720, 720, 32])
INFO: pair_features_list[0] shape: torch.Size([1, 720, 720, 1])
INFO: pair_features_list[1] shape: torch.Size([1, 720, 720, 1])
INFO: pair_features_list[2] shape: torch.Size([1, 720, 720, 32])


Memory at Before forward pass (sample 4/5): Allocated: 33.1MB, Reserved: 44.0MB, Free: 12565.8MB, Total: 15931.9MB
Memory at After forward pass (sample 4/5): Allocated: 35.0MB, Reserved: 4908.0MB, Free: 7701.8MB, Total: 15931.9MB
Memory at Before forward pass (sample 5/5): Allocated: 33.1MB, Reserved: 44.0MB, Free: 12569.8MB, Total: 15931.9MB
Memory at After forward pass (sample 5/5): Allocated: 35.0MB, Reserved: 4908.0MB, Free: 7705.8MB, Total: 15931.9MB


INFO: pairing_probs shape: torch.Size([1, 124, 124])
INFO: coupling_matrix shape: torch.Size([1, 124, 124])
INFO: rel_pos_batch shape: torch.Size([1, 124, 124, 32])
INFO: pair_features_list[0] shape: torch.Size([1, 124, 124, 1])
INFO: pair_features_list[1] shape: torch.Size([1, 124, 124, 1])
INFO: pair_features_list[2] shape: torch.Size([1, 124, 124, 32])
INFO: pairing_probs shape: torch.Size([1, 124, 124])
INFO: coupling_matrix shape: torch.Size([1, 124, 124])
INFO: rel_pos_batch shape: torch.Size([1, 124, 124, 32])
INFO: pair_features_list[0] shape: torch.Size([1, 124, 124, 1])
INFO: pair_features_list[1] shape: torch.Size([1, 124, 124, 1])
INFO: pair_features_list[2] shape: torch.Size([1, 124, 124, 32])
INFO: pairing_probs shape: torch.Size([1, 124, 124])
INFO: coupling_matrix shape: torch.Size([1, 124, 124])
INFO: rel_pos_batch shape: torch.Size([1, 124, 124, 32])
INFO: pair_features_list[0] shape: torch.Size([1, 124, 124, 1])
INFO: pair_features_list[1] shape: torch.Size([1, 124, 

Memory at After batch 8/12: Allocated: 29.1MB, Reserved: 44.0MB, Free: 12569.8MB, Total: 15931.9MB
Memory at Before batch 9/12: Allocated: 29.1MB, Reserved: 42.0MB, Free: 12571.8MB, Total: 15931.9MB
Memory at Before processing batch of size 1: Allocated: 29.1MB, Reserved: 42.0MB, Free: 12571.8MB, Total: 15931.9MB
INFO: lengths shape: torch.Size([1])
INFO: sequence_int shape: torch.Size([1, 124])
INFO: dihedral_features shape: torch.Size([1, 124, 4])
INFO: dihedral_targets shape: torch.Size([1, 124, 4])
INFO: pairing_probs shape: torch.Size([1, 124, 124])
INFO: positional_entropy shape: torch.Size([1, 124])
INFO: accessibility shape: torch.Size([1, 124])
INFO: coupling_matrix shape: torch.Size([1, 124, 124])
INFO: conservation shape: torch.Size([1, 124])
INFO: mask shape: torch.Size([1, 124])
Memory at Before forward pass (sample 1/5): Allocated: 29.2MB, Reserved: 42.0MB, Free: 12571.8MB, Total: 15931.9MB
Memory at After forward pass (sample 1/5): Allocated: 31.2MB, Reserved: 86.0MB, Fr

## 5. Format for Kaggle Submission

Format the results according to Kaggle submission requirements

In [7]:
def format_kaggle_submission(results):
    """Format results for Kaggle submission.
    
    Args:
        results: Dictionary mapping target_id to list of sample dictionaries
        
    Returns:
        Pandas DataFrame formatted according to Kaggle submission guidelines
    """
    # Check if results are empty
    if not results:
        print("WARNING: No results to format for submission")
        return pd.DataFrame(columns=['target_id', 'model_id', 'coordinates', 'confidence'])
    
    # Prepare submission data
    submission_rows = []
    
    for target_id, samples in results.items():
        # Must have exactly 5 samples per target
        if len(samples) < 5:
            print(f"WARNING: Target {target_id} has only {len(samples)} samples. Need 5.")
            # Duplicate the last sample if needed
            if samples:  # Make sure we have at least one sample to duplicate
                samples = samples + [samples[-1]] * (5 - len(samples))
            else:
                print(f"ERROR: No samples for target {target_id}")
                continue
        elif len(samples) > 5:
            # Keep only the first 5
            samples = samples[:5]
        
        # Create rows for each sample
        for i, sample in enumerate(samples):
            try:
                # Format coordinates as string
                coords_str = json.dumps(sample['coords'].tolist())
                confidence_str = json.dumps(sample['confidence'].tolist())
                
                # Add to submission rows
                submission_rows.append({
                    'target_id': target_id,
                    'model_id': i,
                    'coordinates': coords_str,
                    'confidence': confidence_str,
                })
            except Exception as e:
                print(f"ERROR processing sample {i} for target {target_id}: {str(e)}")
                print(f"Sample keys: {list(sample.keys())}")
    
    # Create DataFrame with intermediate format
    intermediate_df = pd.DataFrame(submission_rows)
    
    # Now convert to the final Kaggle format
    # First, load the test sequences to get residue names
    if is_kaggle:
        sequences_path = "/kaggle/input/stanford-rna-3d-folding/test_sequences.csv"
    else:
        sequences_path = "../data/raw/test_sequences.csv"
    
    try:
        sequences_df = pd.read_csv(sequences_path)
        print(f"Loaded {len(sequences_df)} sequences for reformatting")
        
        # Process the sequence data
        sequence_dict = {}
        for _, row in sequences_df.iterrows():
            target_id = row['target_id']
            sequence = row['sequence']
            sequence_dict[target_id] = sequence
        
        # Parse and organize the coordinate data
        coords_by_target_model = {}
        for _, row in intermediate_df.iterrows():
            target_id = row['target_id']
            model_id = int(row['model_id'])
            
            # Parse coordinates JSON
            coords = json.loads(row['coordinates'])
            
            # Store the coordinates
            if target_id not in coords_by_target_model:
                coords_by_target_model[target_id] = {}
            
            coords_by_target_model[target_id][model_id] = coords
        
        # Reformat for final Kaggle submission
        new_rows = []
        for target_id, sequence in sequence_dict.items():
            if target_id not in coords_by_target_model:
                print(f"Warning: No predictions found for target {target_id}")
                continue
                
            for residue_idx, residue_name in enumerate(sequence):
                # 1-based residue indexing
                residue_pos = residue_idx + 1
                
                # Create the row ID
                row_id = f"{target_id}_{residue_pos}"
                
                # Initialize the row with basic info
                new_row = {
                    'ID': row_id,
                    'resname': residue_name,
                    'resid': residue_pos
                }
                
                # Add coordinates for each model
                for model_id in range(5):  # Always process models 0-4
                    if model_id not in coords_by_target_model[target_id]:
                        print(f"Warning: Model {model_id} missing for {target_id}")
                        # If model missing, use zeros
                        new_row[f'x_{model_id+1}'] = 0.0
                        new_row[f'y_{model_id+1}'] = 0.0
                        new_row[f'z_{model_id+1}'] = 0.0
                    else:
                        # Get coordinates for this residue from this model
                        coords = coords_by_target_model[target_id][model_id]
                        
                        if residue_idx < len(coords):
                            x, y, z = coords[residue_idx]
                            new_row[f'x_{model_id+1}'] = float(x)
                            new_row[f'y_{model_id+1}'] = float(y)
                            new_row[f'z_{model_id+1}'] = float(z)
                        else:
                            print(f"Warning: Residue {residue_pos} out of range for {target_id} model {model_id}")
                            # Handle case where prediction is shorter than sequence
                            new_row[f'x_{model_id+1}'] = 0.0
                            new_row[f'y_{model_id+1}'] = 0.0
                            new_row[f'z_{model_id+1}'] = 0.0
                
                new_rows.append(new_row)
        
        # Create the new submission dataframe with column order matching sample submission
        columns = ['ID', 'resname', 'resid']
        for model_id in range(1, 6):  # Models 1-5 in output
            columns.extend([f'x_{model_id}', f'y_{model_id}', f'z_{model_id}'])
        
        kaggle_df = pd.DataFrame(new_rows, columns=columns)
        print(f"Successfully created Kaggle-format submission with {len(kaggle_df)} rows")
        
        return kaggle_df
    
    except Exception as e:
        print(f"ERROR during submission reformatting: {str(e)}")
        import traceback
        traceback.print_exc()
        
        # Return the intermediate format as fallback
        print("WARNING: Returning intermediate submission format")
        return intermediate_df
    
# Format results for submission
submission_df = format_kaggle_submission(results)
print(f"Submission DataFrame created with {len(submission_df)} rows")

# Display preview
if not submission_df.empty:
    display(submission_df.head())
else:
    print("No submission data to display")

Loaded 12 sequences for reformatting
Successfully created Kaggle-format submission with 2515 rows
Submission DataFrame created with 2515 rows


,ID,resname,resid,x_1,y_1,z_1,x_2,y_2,z_2,x_3,y_3,z_3,x_4,y_4,z_4,x_5,y_5,z_5
0,R1107_1,G,1,-1.442383,-0.335938,-0.449951,-0.797852,-0.314941,-0.802734,-1.584961,0.197510,-0.436035,-1.452148,-0.130859,-0.191040,-1.429688,0.192749,-0.446045
1,R1107_2,G,2,-1.286133,-0.018768,-0.953125,-1.657227,-0.951660,-0.784180,-1.381836,-0.144287,-0.822266,-1.524414,-0.701172,-0.878906,-1.862305,-0.311523,-0.850586
2,R1107_3,G,3,-1.656250,-0.677246,-1.016602,-0.705566,-0.749023,-1.199219,-1.884766,-0.596191,-0.655762,-1.081055,-0.447998,-1.556641,-1.004883,-0.452881,-1.415039
3,R1107_4,G,4,-1.106445,-0.859375,-1.475586,-0.734375,-0.319336,-1.417969,-0.988281,-0.708496,-1.277344,-0.269287,-0.269287,-1.414062,-1.482422,-1.335938,-1.310547
4,R1107_5,G,5,-1.220703,-0.500977,-1.174805,-1.126953,-0.897949,-0.670898,-0.672852,0.246460,-1.330078,-1.305664,0.119995,-0.638184,-0.465332,-0.589355,-1.514648


In [8]:
# For Kaggle, use the standard submission filename
if is_kaggle:
    submission_filename = "submission.csv"
else:
    # In local environment, include timestamp and format indicator
    timestamp = pd.Timestamp.now().strftime("%Y%m%d-%H%M%S")
    model_identifier = "ensemble" if (USE_ENSEMBLE and len(models) > 1) else list(models.keys())[0] if models else "no_model"
    submission_filename = f"submission_kaggle_format_{model_identifier}_{timestamp}.csv"
    
submission_path = os.path.join(OUTPUT_DIR, submission_filename)

# Check if we have data to save
if submission_df.empty:
    print("WARNING: Empty submission DataFrame. No file will be saved.")
else:
    # Save submission file
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    submission_df.to_csv(submission_path, index=False)
    print(f"Submission saved to {submission_path}")

# Also create a submission metadata file with model information
metadata = {
    "timestamp": pd.Timestamp.now().strftime("%Y%m%d-%H%M%S"),
    "environment": "kaggle" if is_kaggle else "local",
    "models_used": list(models.keys()) if models else [],
    "ensemble": USE_ENSEMBLE,
    "temperature": TEMPERATURE,
    "num_samples": NUM_SAMPLES,
    "metrics": {name: info["metrics"] for name, info in models.items()} if models else {},
    "submission_format": "kaggle_residue_format" if 'ID' in submission_df.columns else "intermediate_format",
    "submission_size": len(submission_df),
    "num_targets": len(submission_df['ID'].str.split('_').str[0].unique()) if 'ID' in submission_df.columns else len(submission_df['target_id'].unique()) if not submission_df.empty else 0
}

# Save metadata - use a simple name for Kaggle
if is_kaggle:
    metadata_path = os.path.join(OUTPUT_DIR, "submission_metadata.json")
else:
    model_identifier = "ensemble" if (USE_ENSEMBLE and len(models) > 1) else list(models.keys())[0] if models else "no_model"
    metadata_path = os.path.join(OUTPUT_DIR, f"metadata_{model_identifier}_{metadata['timestamp']}.json")

with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=2)

print(f"Submission metadata saved to {metadata_path}")

# If we're in Kaggle, display a final success message
if is_kaggle:
    print("\n===== KAGGLE SUBMISSION READY =====")
    print(f"Submission file saved to: {submission_path}")
    format_type = "Kaggle residue format" if 'ID' in submission_df.columns else "Intermediate format"
    print(f"Format type: {format_type}")
    if 'ID' in submission_df.columns:
        print(f"Total targets processed: {len(submission_df['ID'].str.split('_').str[0].unique())}")
        print(f"Total residues: {len(submission_df)}")
    else:
        print(f"Total targets processed: {len(submission_df['target_id'].unique())}")
        print(f"Total model samples: {len(submission_df)}")
    print("=====================================")

Submission saved to ../submissions/submission_kaggle_format_final_model_20250423-214240.csv
Submission metadata saved to ../submissions/metadata_final_model_20250423-214240.json


## 6. Generate execution log



In [ ]:
def generate_execution_log():
    """
    Generate a comprehensive execution log from notebook outputs.
    
    This function captures all cell outputs from the Jupyter kernel,
    including errors, and creates a markdown report for debugging.
    
    Improved with robust handling of non-string data types from IPython history.
    """
    from IPython import get_ipython
    from datetime import datetime
    import os
    import json
    import sys
    import traceback
    
    # Get notebook shell
    shell = get_ipython()
    
    # Get all cells and their outputs with proper type checking
    try:
        cells = list(shell.history_manager.get_range())
    except Exception as e:
        print(f"Error getting cell history: {str(e)}")
        cells = []  # Use empty list as fallback
    
    # Generate timestamp
    timestamp = datetime.now().strftime("%Y%m%d-%H%M%S")
    
    # Create markdown report
    report = [
        f"# Kaggle Inference Notebook Execution Log",
        f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}",
        "",
        "## Environment Information",
        f"Device used: {device}",
        f"Python version: {sys.version}",
        f"PyTorch version: {torch.__version__}",
        "",
        "## Configuration",
        "```python",
    ]
    
    # Add configuration details
    configs = {
        "MODEL_PATHS": MODEL_PATHS,
        "SELECTED_MODEL": SELECTED_MODEL,
        "USE_ENSEMBLE": USE_ENSEMBLE,
        "BATCH_SIZE": BATCH_SIZE,
        "NUM_SAMPLES": NUM_SAMPLES,
        "TEMPERATURE": TEMPERATURE
    }
    
    report.append(json.dumps(configs, indent=2))
    report.append("```")
    report.append("")
    
    # Add cell outputs
    report.append("## Cell Execution Results")
    
    # Keep track of cell execution count
    cell_count = 1
    
    # Get the user namespace to execute code and retrieve variables
    user_ns = shell.user_ns
    
    # Log the number of cells being processed
    print(f"Processing {len(cells)} cells from history")
    
    # Iterate through executed cells with robust type handling
    for session, line, exec_count in cells:
        # Robust handling of non-string content in history
        if not isinstance(line, str):
            line_str = f"[Non-string content of type {type(line).__name__}]"
            print(f"Skipping non-string content in cell history: {line_str}")
            continue
        
        # Skip empty lines and the log generator itself
        line = line.strip()
        if not line or "generate_execution_log" in line:
            continue
        
        # Add cell input
        report.append(f"### Cell {cell_count}: Input")
        report.append("```python")
        report.append(line)
        report.append("```")
        
        # Try to capture the output
        report.append(f"### Cell {cell_count}: Output")
        report.append("```")
        
        # Check if variables were assigned in this cell by inspecting the code
        var_names = []
        if "=" in line and not line.lstrip().startswith(("if", "for", "while", "def", "class")):
            # Simple variable assignment detection
            parts = line.split("=")[0].strip().split()
            if parts:
                var_name = parts[-1]
                # Clean up the variable name (remove trailing characters)
                var_name = var_name.strip("()[]{},.;:")
                var_names.append(var_name)
        
        # For each potential variable, try to grab its value from user namespace
        for var_name in var_names:
            if var_name in user_ns:
                try:
                    var_value = user_ns[var_name]
                    # For DataFrames, show a preview
                    if 'pandas.core.frame.DataFrame' in str(type(var_value)):
                        report.append(f"{var_name} (shape: {var_value.shape}):")
                        report.append(str(var_value.head()))
                    # For dictionaries with simple values, show content
                    elif isinstance(var_value, dict) and len(var_value) < 10:
                        report.append(f"{var_name} (dict with {len(var_value)} items):")
                        report.append(str(var_value))
                    # For lists with reasonable size
                    elif isinstance(var_value, list) and len(var_value) < 10:
                        report.append(f"{var_name} (list with {len(var_value)} items):")
                        report.append(str(var_value))
                    else:
                        report.append(f"{var_name}: {type(var_value).__name__} created")
                except Exception as e:
                    report.append(f"Error retrieving value for {var_name}: {str(e)}")
                    
        report.append("```")
        report.append("")
        
        cell_count += 1
    
    # Add a section for errors with line numbers from the running session
    report.append("## Errors Detected")
    
    # Try to get the current IPython instance's error history
    ip = get_ipython()
    
    if hasattr(ip, '_last_traceback'):
        report.append("Most recent error traceback:")
        report.append("```")
        try:
            last_error = "".join(traceback.format_tb(ip._last_traceback))
            report.append(last_error)
        except Exception as e:
            report.append(f"Error formatting traceback: {str(e)}")
        report.append("```")
    else:
        report.append("No recent error traceback available.")
    
    # Add model information
    report.append("## Model Information")
    if 'models' in user_ns:
        model_info = user_ns['models']
        report.append(f"Number of models loaded: {len(model_info)}")
        for model_name, info in model_info.items():
            report.append(f"### {model_name}")
            if 'metrics' in info:
                report.append("**Metrics:**")
                for metric_name, metric_value in info['metrics'].items():
                    report.append(f"- {metric_name}: {metric_value}")
            report.append("")
    else:
        report.append("No models loaded or model information not available.")
    
    # Add specific debugging for model loading and inference
    report.append("## Model Loading/Inference Debug Info")
    if 'models' in user_ns:
        report.append(f"models dictionary has {len(user_ns['models'])} entries.")
    else:
        report.append("models variable not found.")
    
    if 'results' in user_ns:
        report.append(f"results dictionary has {len(user_ns['results'])} entries.")
        
        # Sample one result entry if available
        if len(user_ns['results']) > 0:
            first_key = list(user_ns['results'].keys())[0]
            first_samples = user_ns['results'][first_key]
            report.append(f"First target ({first_key}) has {len(first_samples)} samples.")
            
            if len(first_samples) > 0:
                # Check for expected fields in sample
                first_sample = first_samples[0]
                report.append(f"First sample keys: {list(first_sample.keys())}")
                
                # Check shapes for coords and confidence
                if 'coords' in first_sample:
                    report.append(f"coords shape: {first_sample['coords'].shape}")
                if 'confidence' in first_sample:
                    report.append(f"confidence shape: {first_sample['confidence'].shape}")
    else:
        report.append("results variable not found.")
    
    # Add submission information
    report.append("## Submission Information")
    if 'submission_df' in user_ns:
        sub_df = user_ns['submission_df']
        report.append(f"Submission shape: {sub_df.shape}")
        
        # Check for different submission formats
        if 'ID' in sub_df.columns:
            report.append(f"Kaggle format submission")
            report.append(f"Target IDs: {len(sub_df['ID'].str.split('_').str[0].unique())}")
            report.append(f"Total residues: {len(sub_df)}")
            report.append("")
            report.append("**Sample entries:**")
            report.append("```")
            report.append(str(sub_df.head(3)))
            report.append("```")
        elif 'target_id' in sub_df.columns:
            report.append(f"Intermediate format submission")
            report.append(f"Target IDs: {len(sub_df['target_id'].unique())}")
            report.append(f"Model IDs per target: {sub_df.groupby('target_id').size().mean()}")
            report.append("")
            report.append("**Sample entries:**")
            report.append("```")
            report.append(str(sub_df.head(3)))
            report.append("```")
        else:
            report.append(f"Unknown submission format with columns: {sub_df.columns.tolist()}")
    else:
        report.append("No submission DataFrame found.")
    
    # Save the report
    report_text = "\n".join(report)
    
    # Determine the appropriate output directory
    # For Kaggle, use the working directory
    if is_kaggle:
        reports_dir = '/kaggle/working'
    else:
        # In local environment, use notebook_reports directory
        reports_dir = "../notebook_reports"
    
    # Create reports directory if it doesn't exist
    os.makedirs(reports_dir, exist_ok=True)
    
    # Save report
    report_path = os.path.join(reports_dir, f"kaggle_inference_log_{timestamp}.md")
    with open(report_path, "w") as f:
        f.write(report_text)
    
    print(f"Execution log saved to {report_path}")
    return report_path

try:
    # Import traceback directly here to ensure it's available
    import traceback
    
    # Generate the execution log
    execution_log_path = generate_execution_log()
    print(f"Log generation complete! Report saved to {execution_log_path}")
except Exception as e:
    print(f"Error generating log: {str(e)}")
    import traceback  # Import again just to be super safe
    traceback.print_exc()